<a href="https://colab.research.google.com/github/ersin-alt/deneme2/blob/main/temel_haziran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
başka bir kod 3

In [ ]:
yeni kod eklenecek 2

In [ ]:
import re

# Test etmek istediğin her türlü karmaşık metni buraya yapıştırabilirsin
ham_veri = """
BFREN DITAS DOKTA HRKET IZMDC KLRHO KZGYO MAKTK MARKA
BFREN -DITAS- DOKTA- HRKET -IZMDC -KLRHO -KZGYO - MAKTK
"""

# re.findall yöntemiyle metindeki sadece kelimeleri (harf gruplarını) seçiyoruz
# ve set() kullanarak mükerrer (tekrar eden) hisseleri eliyoruz.
hisseler = sorted(list(set(re.findall(r"[A-Z]{3,5}", ham_veri.upper()))))

# Sonuna .IS ekleyip tırnak içine alarak virgülle birleştiriyoruz
formatli_hisseler = ", ".join([f"'{hisse}.IS'" for hisse in hisseler])

print(formatli_hisseler)

In [ ]:
import yfinance as yf
import pandas as pd
import time
import numpy as np
from datetime import datetime, timedelta
import warnings
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import ColorScaleRule, FormulaRule
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

def get_numeric_info(hisse_info, key):
    value = hisse_info.get(key)
    if value is not None:
        try:
            return float(value)
        except ValueError:
            return None
    return None

def profesyonel_tarama(ticker_list):
    results = []

    for symbol in ticker_list:
        try:
            print(f"Veri çekiliyor: {symbol}...")
            hisse = yf.Ticker(symbol)
            info = hisse.info

            # --- Temel Finansal Verileri Çekme ---
            # ROE (Return on Equity)
            roe = info.get('returnOnEquity')
            # Borç/Özkaynak (Debt/Equity)
            de_ratio_raw = info.get('debtToEquity')
            # FAVÖK Marjı (EBITDA Margins)
            ebitda_margin = info.get('ebitdaMargins')
            # F/K Oranı (Trailing P/E Ratio)
            pe_ratio = info.get('trailingPE')
            # PD/DD Oranı (Price/Book Ratio)
            pb_ratio = info.get('priceToBook')
            # Yıllık Kar Büyüme Oranı (Revenue Growth)
            revenue_growth = info.get('revenueGrowth')
            # Serbest Nakit Akışı (Free Cash Flow) - Genellikle son açıklanan dönem için
            free_cashflow = info.get('freeCashflow')
            # Net Kar Marjı (Profit Margins)
            net_margin = info.get('profitMargins')
            # Temettü Verimi (Dividend Yield)
            dividend_yield = info.get('dividendYield')

            # --- Puanlama Sistemi (Toplam Max Puan: ~100-120) ---
            toplam_puan = 0
            analiz_notu = []

            # 1. ROE Puanlama (Max 20 Puan)
            if roe is not None:
                if roe > 0.20: # %20 üzeri
                    toplam_puan += 20
                    analiz_notu.append("ROE: Sektör üstü verimlilik")
                elif roe > 0.10: # %10 üzeri
                    toplam_puan += 10
                    analiz_notu.append("ROE: Ortalama verimlilik")
                else:
                    analiz_notu.append("ROE: Düşük verimlilik")

            # 2. Borç/Özkaynak Puanlama (Max 20 Puan)
            # debtToEquity genellikle yüzde olarak gelir, 100'e bölerek oran elde edelim.
            de_ratio = de_ratio_raw / 100 if de_ratio_raw is not None else 999
            if de_ratio != 999:
                if de_ratio < 1.0: # 1.0 altında
                    toplam_puan += 20
                    analiz_notu.append("Borç/Özkaynak: Güçlü bilanço, düşük borç")
                elif de_ratio < 2.0: # 2.0 altında
                    toplam_puan += 10
                    analiz_notu.append("Borç/Özkaynak: Orta seviye borçluluk")
                else:
                    analiz_notu.append("Borç/Özkaynak: Yüksek borçluluk riski")

            # 3. FAVÖK Marjı Puanlama (Max 15 Puan)
            if ebitda_margin is not None:
                if ebitda_margin > 0.15: # %15 üzeri
                    toplam_puan += 15
                    analiz_notu.append("FAVÖK Marjı: Yüksek operasyonel karlılık")
                elif ebitda_margin > 0.05: # %5 üzeri
                    toplam_puan += 5
                    analiz_notu.append("FAVÖK Marjı: Sıkışık marjlar")

            # 4. F/K Oranı Puanlama (Max 10 Puan)
            if pe_ratio is not None and pe_ratio > 0:
                if pe_ratio < 15: # 15 altında makul kabul edelim
                    toplam_puan += 10
                    analiz_notu.append("F/K: Makul değerleme")
                elif pe_ratio < 25: # 25 altında orta seviye
                    toplam_puan += 5
                    analiz_notu.append("F/K: Ortalama değerleme")
                else:
                    analiz_notu.append("F/K: Yüksek değerleme")

            # 5. PD/DD Oranı Puanlama (Max 10 Puan) - Bankalar için farklılık gösterebilir
            if pb_ratio is not None and pb_ratio > 0:
                if 1.0 <= pb_ratio <= 3.0: # 1 ile 3 arası ideal
                    toplam_puan += 10
                    analiz_notu.append("PD/DD: Dengeli piyasa değeri/defter değeri")
                elif pb_ratio < 1.0: # 1 altında, potansiyel ucuzluk veya sorun
                    toplam_puan += 5
                    analiz_notu.append("PD/DD: Potansiyel düşük değerleme")
                else:
                    analiz_notu.append("PD/DD: Yüksek piyasa değeri/defter değeri")

            # 6. Kar Büyüme Oranı Puanlama (Max 10 Puan) - Son yıllık büyüme kullanılıyor
            if revenue_growth is not None:
                if revenue_growth > 0.10: # %10 üzeri
                    toplam_puan += 10
                    analiz_notu.append("Gelir Büyümesi: Yüksek büyüme potansiyeli")
                elif revenue_growth > 0.05: # %5 üzeri
                    toplam_puan += 5
                    analiz_notu.append("Gelir Büyümesi: Orta seviye büyüme")
                else:
                    analiz_notu.append("Gelir Büyümesi: Düşük büyüme")

            # 7. Serbest Nakit Akışı Puanlama (Max 10 Puan)
            if free_cashflow is not None and free_cashflow > 0:
                toplam_puan += 10
                analiz_notu.append("Serbest Nakit Akışı: Pozitif ve sağlıklı nakit akışı")
            elif free_cashflow is not None and free_cashflow <= 0:
                analiz_notu.append("Serbest Nakit Akışı: Negatif veya zayıf nakit akışı")

            # 8. Net Kar Marjı Puanlama (Max 10 Puan)
            if net_margin is not None:
                if net_margin > 0.10: # %10 üzeri
                    toplam_puan += 10
                    analiz_notu.append("Net Kar Marjı: Yüksek karlılık")
                elif net_margin > 0.05: # %5 üzeri
                    toplam_puan += 5
                    analiz_notu.append("Net Kar Marjı: Orta seviye karlılık")
                else:
                    analiz_notu.append("Net Kar Marjı: Düşük karlılık")

            # 9. Temettü Verimi Puanlama (Max 5 Puan) - Bonus puan, temettü odaklı yatırımcılar için
            if dividend_yield is not None:
                if 0.03 <= dividend_yield <= 0.06: # %3 ile %6 arası
                    toplam_puan += 5
                    analiz_notu.append("Temettü: Cazip temettü verimi")
                elif dividend_yield > 0.06: # %6 üzeri, potansiyel risk veya yüksek temettü hissesi
                    toplam_puan += 3
                    analiz_notu.append("Temettü: Yüksek temettü verimi")
                else:
                    analiz_notu.append("Temettü: Düşük/Yok temettü")

            results.append({
                "Hisse": symbol,
                "Puan": toplam_puan,
                "ROE (%)": round(roe*100, 2) if roe is not None else "N/A",
                "Borç/Özkaynak": round(de_ratio, 2) if de_ratio != 999 else "N/A",
                "FAVÖK Marjı (%)": round(ebitda_margin*100, 2) if ebitda_margin is not None else "N/A",
                "F/K": round(pe_ratio, 2) if pe_ratio is not None else "N/A",
                "PD/DD": round(pb_ratio, 2) if pb_ratio is not None else "N/A",
                "Gelir Büyümesi (%)": round(revenue_growth*100, 2) if revenue_growth is not None else "N/A",
                "Serbest Nakit Akışı": f"{free_cashflow:,.0f}" if free_cashflow is not None else "N/A",
                "Net Kar Marjı (%)": round(net_margin*100, 2) if net_margin is not None else "N/A",
                "Temettü Verimi (%)": round(dividend_yield*100, 2) if dividend_yield is not None else "N/A",
                "Analiz Notu": "; ".join(analiz_notu)
            })

            time.sleep(0.5) # Yahoo kısıtlamasından kaçınmak için bekleme süresi artırıldı

        except Exception as e:
            print(f"{symbol} hatası: {e}")
            time.sleep(0.5) # Hata durumunda da bekle
            continue

    return pd.DataFrame(results)

# Hisse Listesi - Genişletilmiş liste, daha uzun sürebilir.
# Not: Yahoo Finance her zaman tüm semboller için veri sağlamayabilir, bu durumda 404 hatası alınabilir.
hisseler = [ "ERCB.IS"
      ]

def save_dataframe_to_excel_with_formatting(df, filename):

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Rapor", startrow=1)

        ws = writer.sheets["Rapor"]
        wb = writer.book

        # Header Row Styling
        baslik_font = Font(bold=True, size=12, color="FFFFFF")
        baslik_dolu = PatternFill("solid", fgColor="2E4057")
        orta = Alignment(horizontal="center", vertical="center")
        ince_kenar = Border(
            left=Side(style="thin"), right=Side(style="thin"),
            top=Side(style="thin"), bottom=Side(style="thin")
        )

        ws.row_dimensions[1].height = 8
        ws.row_dimensions[2].height = 22

        sutunlar_rapor = df.columns.tolist()
        genislik_rapor = {
            "Hisse": 15, "Puan": 10, "ROE (%)": 15, "Borç/Özkaynak": 18,
            "FAVÖK Marjı (%)": 18, "F/K": 10, "PD/DD": 10,
            "Gelir Büyümesi (%)": 18, "Serbest Nakit Akışı": 22,
            "Net Kar Marjı (%)": 18, "Temettü Verimi (%)": 18,
            "Analiz Notu": 60
        }

        for col_idx, col_name in enumerate(sutunlar_rapor, 1):
            char = get_column_letter(col_idx)
            ws.column_dimensions[char].width = genislik_rapor.get(col_name, 15)
            cell = ws.cell(row=2, column=col_idx)
            cell.value = col_name
            cell.font = baslik_font
            cell.fill = baslik_dolu
            cell.alignment = orta
            cell.border = ince_kenar

        # Data rows formatting
        acik_gri = PatternFill("solid", fgColor="F5F5F5")
        beyaz = PatternFill("solid", fgColor="FFFFFF")

        for row_idx, row in enumerate(
            ws.iter_rows(min_row=3, max_row=2 + len(df), min_col=1, max_col=len(sutunlar_rapor)),
            0
        ):
            for cell in row:
                cell.alignment = orta
                cell.border = ince_kenar
                cell.fill = acik_gri if row_idx % 2 == 0 else beyaz

        # Conditional Formatting for 'Puan' column (Red-Yellow-Green Scale)
        if "Puan" in df.columns:
            puan_col_idx = df.columns.get_loc("Puan") + 1
            puan_col_letter = get_column_letter(puan_col_idx)
            puan_range = f"{puan_col_letter}3:{puan_col_letter}{2 + len(df)}"
            ws.conditional_formatting.add(
                puan_range,
                ColorScaleRule(
                    start_type='num', start_value=0, start_color='FFC7CE', # Light Red
                    mid_type='num', mid_value=50, mid_color='FFFFCC',    # Light Yellow
                    end_type='num', end_value=100, end_color='C6EFCE'   # Light Green
                )
            )

            # Conditional Formatting for 'Hisse' column based on 'Puan'
            if "Hisse" in df.columns:
                hisse_col_idx = df.columns.get_loc("Hisse") + 1
                hisse_col_letter = get_column_letter(hisse_col_idx)
                hisse_range = f"{hisse_col_letter}3:{hisse_col_letter}{2 + len(df)}"

                # Green for 100 points
                ws.conditional_formatting.add(hisse_range,
                    FormulaRule(formula=[f'${puan_col_letter}3=100'],
                                fill=PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid'))
                )
                # Red for 0 points
                ws.conditional_formatting.add(hisse_range,
                    FormulaRule(formula=[f'${puan_col_letter}3=0'],
                                fill=PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid'))
                )
                # Yellow for scores between 50 and 99
                ws.conditional_formatting.add(hisse_range,
                    FormulaRule(formula=[f'AND(${puan_col_letter}3>=50,${puan_col_letter}3<=99)'],
                                fill=PatternFill(start_color='FFFFCC', end_color='FFFFCC', fill_type='solid'))
                )
                # Orange for scores between 1 and 49
                ws.conditional_formatting.add(hisse_range,
                    FormulaRule(formula=[f'AND(${puan_col_letter}3>=1,${puan_col_letter}3<=49)'],
                                fill=PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid'))
                )

        # Conditional Formatting for ratio columns
        ratio_cols = {
            "ROE (%)": {"better": "higher", "min_val": 0, "max_val": 20},
            "FAVÖK Marjı (%)": {"better": "higher", "min_val": 0, "max_val": 15},
            "Gelir Büyümesi (%)": {"better": "higher", "min_val": 0, "max_val": 10},
            "Net Kar Marjı (%)": {"better": "higher", "min_val": 0, "max_val": 10},
            "Temettü Verimi (%)": {"better": "higher", "min_val": 0, "max_val": 6},
            "Borç/Özkaynak": {"better": "lower", "min_val": 0, "max_val": 2},
            "F/K": {"better": "lower", "min_val": 0, "max_val": 25},
            "PD/DD": {"better": "ideal_range", "ideal_min": 1, "ideal_max": 3}
        }

        for col_name, config in ratio_cols.items():
            if col_name in df.columns:
                col_idx = df.columns.get_loc(col_name) + 1
                col_letter = get_column_letter(col_idx)
                data_range = f"{col_letter}3:{col_letter}{2 + len(df)}"

                # Convert to numeric, coercing errors to NaN
                numeric_col = pd.to_numeric(df[col_name], errors='coerce')

                # Handle cases where min/max might be NaN for empty columns after coercion
                min_val_data = numeric_col.min()
                max_val_data = numeric_col.max()

                if config["better"] == "higher":
                    ws.conditional_formatting.add(
                        data_range,
                        ColorScaleRule(
                            start_type='num', start_value=min_val_data if not pd.isna(min_val_data) else 0, start_color='FFC7CE', # Light Red
                            mid_type='num', mid_value=(max_val_data + (min_val_data if not pd.isna(min_val_data) else 0))/2 if not pd.isna(min_val_data) and not pd.isna(max_val_data) else 0, mid_color='FFFFCC', # Light Yellow
                            end_type='num', end_value=max_val_data if not pd.isna(max_val_data) else 100, end_color='C6EFCE' # Light Green
                        )
                    )
                elif config["better"] == "lower":
                    ws.conditional_formatting.add(
                        data_range,
                        ColorScaleRule(
                            start_type='num', start_value=min_val_data if not pd.isna(min_val_data) else 0, start_color='C6EFCE', # Light Green
                            mid_type='num', mid_value=(max_val_data + (min_val_data if not pd.isna(min_val_data) else 0))/2 if not pd.isna(min_val_data) and not pd.isna(max_val_data) else 0, mid_color='FFFFCC', # Light Yellow
                            end_type='num', end_value=max_val_data if not pd.isna(max_val_data) else 100, end_color='FFC7CE' # Light Red
                        )
                    )
                elif config["better"] == "ideal_range":
                    # For PD/DD, highlight ideal range (green) and outside (red)
                    ideal_min = config["ideal_min"]
                    ideal_max = config["ideal_max"]

                    # Values below ideal_min (red)
                    ws.conditional_formatting.add(data_range,
                        FormulaRule(formula=[f'{col_letter}3<{ideal_min}'],
                                    fill=PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid'))
                    )
                    # Values within ideal_range (green)
                    ws.conditional_formatting.add(data_range,
                        FormulaRule(formula=[f'AND({col_letter}3>={ideal_min},{col_letter}3<={ideal_max})'],
                                    fill=PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid'))
                    )
                    # Values above ideal_max (red)
                    ws.conditional_formatting.add(data_range,
                        FormulaRule(formula=[f'{col_letter}3>{ideal_max}'],
                                    fill=PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid'))
                    )

    print(f"\n✅ Kaydedildi: {filename}")


# İşlemi Başlat
rapor = profesyonel_tarama(hisseler)

# Sıralı Şekilde Ekrana Yazdır
print("\n--- TARAMA TAMAMLANDI ---")
# pd.set_option('display.max_columns', None) # Tüm sütunları göstermek için
# pd.set_option('display.width', 1000) # Geniş ekran için
print(rapor.sort_values(by="Puan", ascending=False).to_string())

# Excel Dosyasına Kaydet
save_dataframe_to_excel_with_formatting(rapor.sort_values(by="Puan", ascending=False), "temel_analiz_nihai.xlsx")
print("\nSonuçlar 'temel_analiz_nihai.xlsx' dosyasına kaydedildi.")

Veri çekiliyor: ERCB.IS...

--- TARAMA TAMAMLANDI ---
     Hisse  Puan  ROE (%)  Borç/Özkaynak  FAVÖK Marjı (%)  F/K  PD/DD  Gelir Büyümesi (%) Serbest Nakit Akışı  Net Kar Marjı (%) Temettü Verimi (%)                                                                                                                                                                                                                                              Analiz Notu
0  ERCB.IS    25    -4.38           1.11             5.06  N/A   1.47               -27.2        -483,046,720              -3.06                N/A  ROE: Düşük verimlilik; Borç/Özkaynak: Orta seviye borçluluk; FAVÖK Marjı: Sıkışık marjlar; PD/DD: Dengeli piyasa değeri/defter değeri; Gelir Büyümesi: Düşük büyüme; Serbest Nakit Akışı: Negatif veya zayıf nakit akışı; Net Kar Marjı: Düşük karlılık

✅ Kaydedildi: temel_analiz_nihai.xlsx

Sonuçlar 'temel_analiz_nihai.xlsx' dosyasına kaydedildi.
